# Local Rerun 3: Build Corrected Panel and Final Sample Flow

This notebook rebuilds the merged TRC + ERA5 panel for the cleanly screened city list,
applies full-period retention on the *actual* 2003-2025 panel, and defines the final
`high_veg` subgroup transparently.


In [ ]:
from pathlib import Path
from glob import glob
import pandas as pd
import numpy as np

ROOT = Path("/Users/apple/Downloads")
OUT = Path("/Users/apple/Downloads/nbs/rerun_repair/outputs")
OUT.mkdir(parents=True, exist_ok=True)

TRC_GLOB = str(ROOT / "UCDB_TRC_*_screened_fast.csv")
ERA_GLOB = str(ROOT / "ERA5L_daily_*_screened_core.csv")
RETENTION_2025 = ROOT / "UCDB_TRC_2025_city_retention_all11422.csv"

MIN_PIXELS = 5
FULL_PERIOD_FRAC = 0.50


In [ ]:
trc_files = sorted(glob(TRC_GLOB))
era_files = sorted(glob(ERA_GLOB))
print("TRC files:", len(trc_files))
print("ERA files:", len(era_files))

if len(trc_files) == 0 or len(era_files) == 0:
    raise FileNotFoundError(
        "Expected yearly screened TRC and ERA5 files. Run notebook 03 in Colab first."
    )


In [ ]:
trc = pd.concat([pd.read_csv(f, dtype={"uc_id": str}) for f in trc_files], ignore_index=True)
trc["date"] = pd.to_datetime(trc["date"])
trc["year"] = trc["date"].dt.year
trc["doy"] = trc["date"].dt.dayofyear
trc["step8"] = trc.groupby(["uc_id", "unit", "year"]).cumcount() + 1

trc["dT_day_qc"] = np.where(
    (trc["n_built_day"] >= MIN_PIXELS) & (trc["n_veg_day"] >= MIN_PIXELS),
    trc["dT_day"],
    np.nan
)
trc["dT_night_qc"] = np.where(
    (trc["n_built_night"] >= MIN_PIXELS) & (trc["n_veg_night"] >= MIN_PIXELS),
    trc["dT_night"],
    np.nan
)

trc["dT_day_qc"] = trc["dT_day_qc"].where(trc["dT_day_qc"].between(-15, 15))
trc["dT_night_qc"] = trc["dT_night_qc"].where(trc["dT_night_qc"].between(-10, 10))

trc_p = (
    trc.groupby(["uc_id", "unit", "year", "step8"], as_index=False)
       .agg(
           date_min=("date", "min"),
           date_max=("date", "max"),
           dT_day=("dT_day_qc", "mean"),
           dT_night=("dT_night_qc", "mean"),
           tb_day=("tb_day", "mean"),
           tb_night=("tb_night", "mean"),
           tv_day=("tv_day", "mean"),
           tv_night=("tv_night", "mean"),
           n_obs_day=("dT_day_qc", lambda x: x.notna().sum()),
           n_obs_night=("dT_night_qc", lambda x: x.notna().sum()),
           n_built_day=("n_built_day", "mean"),
           n_built_night=("n_built_night", "mean"),
           n_veg_day=("n_veg_day", "mean"),
           n_veg_night=("n_veg_night", "mean"),
       )
)
print(trc_p.shape)


In [ ]:
era = pd.concat([pd.read_csv(f, dtype={"uc_id": str}) for f in era_files], ignore_index=True)
era["date"] = pd.to_datetime(era["date"])
era["year"] = era["date"].dt.year
era["doy"] = era["date"].dt.dayofyear
era["pentad"] = ((era["doy"] - 1) // 5) + 1

era["t2m_c"] = era["temperature_2m"] - 273.15
era["td2m_c"] = era["dewpoint_temperature_2m"] - 273.15
era["skin_c"] = era["skin_temperature"] - 273.15

def sat_vp(temp_c):
    return 0.6108 * np.exp((17.27 * temp_c) / (temp_c + 237.3))

era["es"] = sat_vp(era["t2m_c"])
era["ea"] = sat_vp(era["td2m_c"])
era["vpd_kpa"] = era["es"] - era["ea"]

era_p = (
    era.groupby(["uc_id", "year", "pentad"], as_index=False)
       .agg(
           t2m_c=("t2m_c", "mean"),
           td2m_c=("td2m_c", "mean"),
           skin_c=("skin_c", "mean"),
           rzsm=("rzsm", "mean"),
           vpd_kpa=("vpd_kpa", "mean"),
           pr_sum=("total_precipitation_sum", "sum"),
       )
)

if era_p["pentad"].nunique() and trc_p["step8"].nunique():
    step_lookup = (
        trc_p[["year", "step8", "date_min"]]
        .drop_duplicates()
        .sort_values(["year", "step8"])
    )
    step_lookup["pentad"] = step_lookup["date_min"].dt.dayofyear.sub(1).floordiv(5).add(1)
    step_lookup = step_lookup[["year", "step8", "pentad"]].drop_duplicates()
    era_p = era_p.merge(step_lookup, on=["year", "pentad"], how="inner")
else:
    raise ValueError("Could not align ERA pentads with TRC 8-day steps.")


In [ ]:
merged = trc_p.merge(
    era_p.drop(columns=["pentad"]),
    on=["uc_id", "year", "step8"],
    how="left"
)

core = merged[merged["unit"] == "core"].copy()
core["rzsm_pct"] = core.groupby(["uc_id", "step8"])["rzsm"].transform(lambda x: x.rank(pct=True, method="average") * 100.0)
core["vpd_pct"] = core.groupby(["uc_id", "step8"])["vpd_kpa"].transform(lambda x: x.rank(pct=True, method="average") * 100.0)
core["t2m_pct"] = core.groupby(["uc_id", "step8"])["t2m_c"].transform(lambda x: x.rank(pct=True, method="average") * 100.0)

merged = merged.drop(columns=[c for c in ["rzsm_pct", "vpd_pct", "t2m_pct"] if c in merged.columns], errors="ignore")
merged = merged.merge(
    core[["uc_id", "year", "step8", "rzsm_pct", "vpd_pct", "t2m_pct"]].drop_duplicates(),
    on=["uc_id", "year", "step8"],
    how="left"
)
print(merged.shape)


In [ ]:
city_unit_cov = (
    merged.groupby(["uc_id", "unit"], as_index=False)
          .agg(
              day_valid_frac=("dT_day", lambda x: x.notna().mean()),
              night_valid_frac=("dT_night", lambda x: x.notna().mean()),
          )
)

wide = city_unit_cov.pivot(index="uc_id", columns="unit", values=["day_valid_frac", "night_valid_frac"])
wide.columns = ["_".join(col) for col in wide.columns]
wide = wide.reset_index()

wide["retain_day_fullperiod"] = (
    (wide["day_valid_frac_core"] >= FULL_PERIOD_FRAC) &
    (wide["day_valid_frac_ring10_20"] >= FULL_PERIOD_FRAC)
)
wide["retain_night_fullperiod"] = (
    (wide["night_valid_frac_core"] >= FULL_PERIOD_FRAC) &
    (wide["night_valid_frac_ring10_20"] >= FULL_PERIOD_FRAC)
)
wide["retain_either_fullperiod"] = wide["retain_day_fullperiod"] | wide["retain_night_fullperiod"]

keep_ids = set(wide.loc[wide["retain_either_fullperiod"], "uc_id"].astype(str))
panel = merged[merged["uc_id"].isin(keep_ids)].copy()

flow = [
    {"stage": "screened_2025_ids", "cities": int(pd.read_csv(RETENTION_2025, dtype={'uc_id': str})["city_retain_either"].astype(str).str.lower().isin(['true','1','yes']).sum())},
    {"stage": "fullperiod_retained", "cities": len(keep_ids)},
]

city_base = (
    panel[panel["unit"] == "ring10_20"]
    .groupby("uc_id", as_index=False)
    .agg(mean_ring_veg=("n_veg_day", "mean"))
)
med = city_base["mean_ring_veg"].median()
city_base["veg_group"] = np.where(city_base["mean_ring_veg"] >= med, "high_veg", "low_veg")
panel = panel.merge(city_base[["uc_id", "mean_ring_veg", "veg_group"]], on="uc_id", how="left")

flow.append({"stage": "high_veg_main_subgroup", "cities": int((city_base["veg_group"] == "high_veg").sum())})
flow_df = pd.DataFrame(flow)
flow_df


In [ ]:
clim = (
    panel.groupby(["uc_id", "unit", "step8"], as_index=False)
         .agg(
             dT_day_clim=("dT_day", "mean"),
             dT_night_clim=("dT_night", "mean"),
             tb_day_clim=("tb_day", "mean"),
             tb_night_clim=("tb_night", "mean"),
             tv_day_clim=("tv_day", "mean"),
             tv_night_clim=("tv_night", "mean"),
         )
)
panel = panel.merge(clim, on=["uc_id", "unit", "step8"], how="left")
panel["dT_day_anom"] = panel["dT_day"] - panel["dT_day_clim"]
panel["dT_night_anom"] = panel["dT_night"] - panel["dT_night_clim"]
panel["tb_day_anom"] = panel["tb_day"] - panel["tb_day_clim"]
panel["tb_night_anom"] = panel["tb_night"] - panel["tb_night_clim"]
panel["tv_day_anom"] = panel["tv_day"] - panel["tv_day_clim"]
panel["tv_night_anom"] = panel["tv_night"] - panel["tv_night_clim"]

panel.to_csv(OUT / "TRC_threshold_analysis_panel_anom_corrected.csv", index=False)
wide.to_csv(OUT / "city_retention_fullperiod_corrected.csv", index=False)
flow_df.to_csv(OUT / "sample_flow_corrected.csv", index=False)

print(OUT / "TRC_threshold_analysis_panel_anom_corrected.csv")
print(flow_df)
